In [102]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [103]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [104]:
from qiskit_aer import AerSimulator

backend = AerSimulator()

def quantum_random_bits(n):
    """Generate n random bits by measuring n qubits prepared in |+⟩ = H|0⟩."""
    qc = QuantumCircuit(n, n)
    for i in range(n):
        qc.h(i)          # |0⟩ → 1/√2(|0⟩ + |1⟩)
    qc.measure(range(n), range(n))
    # Run directly, skip transpile to avoid the coupling-map qubit limit
    job = backend.run(qc, shots=1, memory=True)
    bits_str = job.result().get_memory()[0]
    # Qiskit stores qubit 0 at the rightmost character, so reverse
    return [int(b) for b in reversed(bits_str)]


In [105]:
N_BITS = 100  # number of qubits Alice transmits

# ALICE
alice_bits  = quantum_random_bits(N_BITS)   # secret bit values to encode
alice_bases = quantum_random_bits(N_BITS)   # 0 = S (standard), 1 = D (diagonal)

print("=== ALICE ===")
print(f"Bits  : {alice_bits[:]}")
print(f"Bases : {alice_bases[:]}")
print("  (basis 0 = S/standard: |0⟩,|1⟩   basis 1 = D/diagonal: |+⟩,|−⟩)")


=== ALICE ===
Bits  : [0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1]
Bases : [0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0]
  (basis 0 = S/standard: |0⟩,|1⟩   basis 1 = D/diagonal: |+⟩,|−⟩)


In [106]:
def eve_intercept(alice_bit, alice_basis, eve_basis):
    """
    Eve intercepts Alice's qubit and measures it in eve_basis.
    Returns the bit Eve observed — she will re-prepare this qubit for Bob.

    When eve_basis != alice_basis, the measurement disturbs the qubit state,
    introducing errors that Alice and Bob can detect later.
    """
    qc = QuantumCircuit(1, 1)
    # Alice's prepared state
    if alice_bit == 1:
        qc.x(0)
    if alice_basis == 1:
        qc.h(0)
    # Eve measures in her chosen basis
    if eve_basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    job = backend.run(qc, shots=1, memory=True)
    return int(job.result().get_memory()[0])

# ---- EVE ----
eve_bases   = quantum_random_bits(N_BITS)
eve_results = [eve_intercept(alice_bits[i], alice_bases[i], eve_bases[i])
               for i in range(N_BITS)]

print("EVE (attacker)")
print(f"Bases   : {eve_bases[:]}")
print(f"Results : {eve_results[:]}")
print("Eve has intercepted all qubits and will re-send them based on her measurements.")


EVE (attacker)
Bases   : [1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0]
Results : [0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1]
Eve has intercepted all qubits and will re-send them based on her measurements.


In [107]:
def bob_measure_forwarded(eve_result, eve_basis, bob_basis):
    """
    Bob measures the qubit that Eve re-prepared from her intercepted result.
    Eve re-sends in her own basis using her measured bit value.
    """
    qc = QuantumCircuit(1, 1)
    # Eve re-prepares her interception result in her basis
    if eve_result == 1:
        qc.x(0)
    if eve_basis == 1:
        qc.h(0)
    # Bob measures in his chosen basis
    if bob_basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    job = backend.run(qc, shots=1, memory=True)
    return int(job.result().get_memory()[0])

# ---- BOB ----
bob_bases   = quantum_random_bits(N_BITS)
bob_results = [bob_measure_forwarded(eve_results[i], eve_bases[i], bob_bases[i])
               for i in range(N_BITS)]

print("BOB")
print(f"Bases   : {bob_bases[:]}")
print(f"Results : {bob_results[:]}")


BOB
Bases   : [1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1]
Results : [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0]


In [108]:
# SIFTING
# Alice and Bob publicly announce their bases.
# They keep only bits where they used the same basis.
print("SIFTING (public basis comparison)")

sifted_alice = []
sifted_bob   = []

for i in range(N_BITS):
    if alice_bases[i] == bob_bases[i]:
        sifted_alice.append(alice_bits[i])
        sifted_bob.append(bob_results[i])

n_sifted = len(sifted_alice)
print(f"Matching bases: {n_sifted} / {N_BITS}")
print(f"Alice sifted key : {sifted_alice[:]}")
print(f"Bob   sifted key : {sifted_bob[:]}")


SIFTING (public basis comparison)
Matching bases: 47 / 100
Alice sifted key : [0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1]
Bob   sifted key : [0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1]


In [109]:
# ATTACK DETECTION
# Alice and Bob publicly compare a sample of their sifted key bits.
# Eve's intercept-resend attack (incorrect basis around 50% of the time) causes around 25% errors
# in the sifted key — well above the 10% threshold. (10% to account for noise)
print("ATTACK DETECTION")

SAMPLE_SIZE   = min(20, n_sifted)
sample_errors = sum(a != b for a, b in zip(sifted_alice[:SAMPLE_SIZE], sifted_bob[:SAMPLE_SIZE]))
error_rate    = sample_errors / SAMPLE_SIZE

print(f"Sample size:  {SAMPLE_SIZE} bits")
print(f"Errors found: {sample_errors}")
print(f"Error rate:   {error_rate:.1%}")
print()
print("(Expected around 25% error rate when a full intercept-resend attack is present)")
print()

THRESHOLD = 0.10   # if >10% errors, eavesdropping suspected 

if error_rate > THRESHOLD:
    print(f"ATTACK DETECTED: error rate {error_rate:.1%} exceeds threshold of {THRESHOLD:.0%}.")
    print("Alice and Bob abort the key exchange, the channel is compromised.")
else:
    print(f"No attack detected (error rate {error_rate:.1%} <= threshold of {THRESHOLD:.0%}).")
    remaining_key = sifted_alice[SAMPLE_SIZE:]
    print(f"\nShared key ({len(remaining_key)} bits):")
    print(''.join(str(b) for b in remaining_key))


ATTACK DETECTION
Sample size:  20 bits
Errors found: 8
Error rate:   40.0%

(Expected around 25% error rate when a full intercept-resend attack is present)

ATTACK DETECTED: error rate 40.0% exceeds threshold of 10%.
Alice and Bob abort the key exchange, the channel is compromised.
